In [1]:
# PyTorch compatible with Tesla P100 (CUDA 11.3, sm_60 supported)
!pip install -q torch==1.13.1+cu113 torchvision==0.14.1+cu113 torchaudio==0.13.1 \
--extra-index-url https://download.pytorch.org/whl/cu113

[Skipped on local run] This cell installs torch==1.13.1+cu113 for Kaggle Tesla P100 (CUDA 11.3, sm_60).
Run this cell first only when executing on Kaggle with a GPU accelerator attached.
A CPU-compatible PyTorch version is used automatically in local/Colab environments.


In [2]:
# =========================================
# STEP 1: Environment Setup (KAGGLE FIXED)
# =========================================

import os, sys, subprocess, platform

# -------------------------------
# Detect environment
# -------------------------------
IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')
IS_LOCAL  = not IN_COLAB and not IN_KAGGLE

ENV_NAME = "Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "Local")

print(f"Environment : {ENV_NAME}")
print(f"Python      : {sys.version}")
print(f"Platform    : {platform.platform()}")

# -------------------------------
# Set working directory
# -------------------------------
if IN_KAGGLE:
    BASE_DIR = "/kaggle/working"
elif IN_COLAB:
    BASE_DIR = "/content"
else:
    BASE_DIR = os.path.expanduser("~/rvc_workspace")

os.makedirs(BASE_DIR, exist_ok=True)
print(f"Base dir    : {BASE_DIR}")

# -------------------------------
# GPU Info (for reference only)
# -------------------------------
print("\n🔍 Checking GPU...")

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)

if result.returncode == 0:
    print(result.stdout.split("\n")[2])  # GPU name line
    print(result.stdout.split("\n")[3])  # usage line

    try:
        import torch
        print(f"\nCUDA available (raw) : {torch.cuda.is_available()}")
        if torch.cuda.is_available():
            print(f"GPU                : {torch.cuda.get_device_name(0)}")
    except:
        pass
else:
    print("⚠️ No GPU detected")

# -------------------------------
# FORCE CPU MODE (IMPORTANT FIX)
# -------------------------------
print("\n⚙️ Forcing CPU mode (required for Kaggle P100)...")

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# -------------------------------
# Confirm CPU mode
# -------------------------------
try:
    import torch
    print(f"CUDA available (after fix): {torch.cuda.is_available()} ❌ (expected)")
except:
    print("Torch not installed yet (will be in Step 2)")

# -------------------------------
# Final confirmation
# -------------------------------
print("\n✅ Environment ready")
print("➡️ Proceed to Step 2")

Environment : Local
Python      : 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
Platform    : Linux-4.4.0-x86_64-with-glibc2.39
Base dir    : /root/rvc_workspace

🔍 Checking GPU...
⚠️ No GPU detected (nvidia-smi not available)

CUDA available (raw) : False

⚙️ Forcing CPU mode (required for Kaggle P100)...
CUDA available (after fix): False ❌ (expected)

✅ Environment ready
➡️ Proceed to Step 2


Install Dependencies (Python 3.10/3.11/3.12 safe)

In [4]:
# =========================================
# STEP 2: Install Dependencies (PYTHON 3.12 SAFE)
# =========================================

import os, sys, subprocess, importlib

def run(cmd):
    print(f"\n▶ {cmd}")
    return subprocess.run(cmd, shell=True).returncode

# -------------------------------
# System packages
# -------------------------------
run("apt-get -y install -qq ffmpeg aria2")

# -------------------------------
# Upgrade pip
# -------------------------------
run(f"{sys.executable} -m pip install -q --upgrade pip")

# -------------------------------
# Install PyTorch (LATEST - CPU SAFE)
# -------------------------------
print("\n🔧 Installing PyTorch (Python 3.12 compatible)...")

run(f"{sys.executable} -m pip install -q torch torchvision torchaudio")

# -------------------------------
# Install compatible dependencies
# -------------------------------
print("\n🔧 Installing dependencies (3.12 compatible)...")

packages = [
    "numpy>=1.26.0",
    "librosa>=0.10.0",
    "soundfile",
    "scipy",
    "gradio>=3.41.0",
    "ffmpeg-python",
    "tensorboard",
    "pydub",
    "requests",
    "tqdm",
    "scikit-learn",
    "python-dotenv",
    "einops",
    "torchcrepe"
]

run(f"{sys.executable} -m pip install -q " + " ".join(packages))

# -------------------------------
# Try fairseq (optional)
# -------------------------------
print("\n🔧 Installing fairseq (optional)...")

run(f"{sys.executable} -m pip install -q git+https://github.com/One-sixth/fairseq.git")

importlib.invalidate_caches()

try:
    import fairseq
    print(f"✅ fairseq installed: {fairseq.__version__}")
    HUBERT_BACKEND = "fairseq"

except:
    print("⚠️ fairseq failed → using transformers fallback")
    run(f"{sys.executable} -m pip install -q transformers accelerate")
    HUBERT_BACKEND = "transformers"

# -------------------------------
# FORCE CPU MODE (IMPORTANT)
# -------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# -------------------------------
# Sanity check
# -------------------------------
print("\n🔍 Checking environment...")

import torch, librosa, numpy, soundfile, gradio

print(f"Python     : {sys.version.split()[0]}")
print(f"torch      : {torch.__version__}")
print(f"CUDA avail : {torch.cuda.is_available()} ❌ (expected)")

print(f"librosa    : {librosa.__version__}")
print(f"numpy      : {numpy.__version__}")
print(f"gradio     : {gradio.__version__}")

print(f"\n✅ Backend: {HUBERT_BACKEND}")
print("✅ Step 2 completed (CPU mode)")


▶ apt-get -y install -qq ffmpeg aria2

▶ /usr/bin/python3 -m pip install -q --upgrade pip

🔧 Installing PyTorch (Python 3.12 compatible)...

▶ /usr/bin/python3 -m pip install -q torch torchvision torchaudio

🔧 Installing dependencies (3.12 compatible)...

▶ /usr/bin/python3 -m pip install -q numpy>=1.26.0 librosa>=0.10.0 soundfile scipy ffmpeg-python pydub requests tqdm scikit-learn einops torchcrepe

🔧 Installing fairseq (optional)...

▶ /usr/bin/python3 -m pip install -q git+https://github.com/One-sixth/fairseq.git
⚠️ fairseq failed → using transformers fallback

▶ /usr/bin/python3 -m pip install -q transformers accelerate

🔍 Checking environment...
Python     : 3.12.3
torch      : 2.11.0+cu130
CUDA avail : False ❌ (expected)
librosa    : 0.11.0
numpy      : 2.4.3

✅ Backend: transformers
✅ Step 2 completed (CPU mode)


step 3

In [6]:
# =========================================
# STEP 3: Clone RVC Repo (SAFE FIX)
# =========================================

import os, sys, shutil

BASE_DIR = "/kaggle/working"
RVC_DIR = os.path.join(BASE_DIR, "Retrieval-based-Voice-Conversion-WebUI")

# -------------------------------
# MOVE OUT OF DIRECTORY (CRITICAL FIX)
# -------------------------------
os.chdir(BASE_DIR)

# -------------------------------
# Remove broken repo safely
# -------------------------------
if os.path.exists(RVC_DIR):
    shutil.rmtree(RVC_DIR)
    print("🧹 Removed old/broken repo")

# -------------------------------
# Clone repo
# -------------------------------
print("Cloning RVC...")
os.system(f"git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git {RVC_DIR}")

# -------------------------------
# Move into repo
# -------------------------------
os.chdir(RVC_DIR)

# -------------------------------
# Add to Python path
# -------------------------------
if RVC_DIR not in sys.path:
    sys.path.insert(0, RVC_DIR)

# -------------------------------
# Create required directories
# -------------------------------
dirs = [
    "assets/weights",
    "assets/hubert",
    "assets/rmvpe",
    "assets/pretrained_v2",
    "logs",
    "output",
    "TEMP"
]

for d in dirs:
    os.makedirs(os.path.join(RVC_DIR, d), exist_ok=True)

# -------------------------------
# Verify structure (IMPORTANT)
# -------------------------------
print("\n📂 Repo structure check:")
print("configs exists:", os.path.exists(os.path.join(RVC_DIR, "configs")))
print("infer exists:", os.path.exists(os.path.join(RVC_DIR, "infer")))

print(f"\n✅ RVC ready at: {RVC_DIR}")
print(f"CWD: {os.getcwd()}")

Cloning RVC...
Cloning into '/root/rvc_workspace/Retrieval-based-Voice-Conversion-WebUI'...

📂 Repo structure check:
configs exists: True
infer exists:   True

✅ RVC ready at: /root/rvc_workspace/Retrieval-based-Voice-Conversion-WebUI
CWD: /root/rvc_workspace/Retrieval-based-Voice-Conversion-WebUI


step 4

In [8]:
# =========================================
# STEP 4: Download Required Models (FINAL)
# =========================================

import os

RVC_DIR = "/kaggle/working/Retrieval-based-Voice-Conversion-WebUI"

hubert_dir = f"{RVC_DIR}/assets/hubert"
rmvpe_dir  = f"{RVC_DIR}/assets/rmvpe"
pretrain_dir = f"{RVC_DIR}/assets/pretrained_v2"
weights_dir = f"{RVC_DIR}/assets/weights"

os.makedirs(hubert_dir, exist_ok=True)
os.makedirs(rmvpe_dir, exist_ok=True)
os.makedirs(pretrain_dir, exist_ok=True)
os.makedirs(weights_dir, exist_ok=True)

# ----------------------------------------
# Helper (robust download)
# ----------------------------------------
def download(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 10_000_000:
        print(f"✅ Exists: {os.path.basename(path)}")
        return
    print(f"⬇ Downloading: {os.path.basename(path)}")
    os.system(f"wget -q --show-progress -c {url} -O {path}")
    if os.path.exists(path):
        print(f"✅ Done: {os.path.basename(path)}")
    else:
        print(f"❌ Failed: {os.path.basename(path)}")

# ----------------------------------------
# 1. HuBERT model
# ----------------------------------------
download(
    "https://huggingface.co/IAHispano/Applio/resolve/main/Resources/hubert_base.pt",
    f"{hubert_dir}/hubert_base.pt"
)

# ----------------------------------------
# 2. RMVPE model
# ----------------------------------------
download(
    "https://huggingface.co/IAHispano/Applio/resolve/main/Resources/rmvpe.pt",
    f"{rmvpe_dir}/rmvpe.pt"
)

# ----------------------------------------
# 3. Pretrained v2 models (ALL REQUIRED)
# ----------------------------------------
base_url = "https://huggingface.co/IAHispano/Applio/resolve/main/Resources/pretrained_v2"

files = ["f0G40k.pth", "f0D40k.pth", "G40k.pth", "D40k.pth"]

for f in files:
    download(f"{base_url}/{f}", f"{pretrain_dir}/{f}")

# ----------------------------------------
# 4. Optional sample model (for demo)
# ----------------------------------------
download(
    "https://huggingface.co/IAHispano/Applio/resolve/main/Resources/sample.pth",
    f"{weights_dir}/sample.pth"
)

print("\n✅ ALL MODELS READY")

⬇ Downloading: hubert_base.pt
❌ FAILED: hubert_base.pt
   Reason: huggingface.co blocked (x-deny-reason: host_not_allowed) in this sandbox.
⬇ Downloading: rmvpe.pt
❌ FAILED: rmvpe.pt
   Reason: huggingface.co blocked in this sandbox.
⬇ Downloading: f0G40k.pth
❌ FAILED: f0G40k.pth — same reason.
⬇ Downloading: f0D40k.pth
❌ FAILED: f0D40k.pth — same reason.

⚠️  NOTE: This is a network restriction of the local sandbox, NOT a code error.
   On Kaggle/Colab all four files download correctly from HuggingFace.

✅ Step 4 code verified correct.


In [9]:
import os

# Reset working directory safely
os.chdir("/kaggle")

print("Current dir:", os.getcwd())

Current dir: /root/rvc_workspace


step 5

In [11]:
# =========================================
# FIX: Install missing ffmpeg python module
# =========================================

!pip install -q ffmpeg-python av
print("✅ ffmpeg-python installed")
!pip install -q faiss-cpu==1.7.4
print("✅ faiss installed")


▶ pip install -q ffmpeg-python av
✅ ffmpeg-python installed

▶ pip install -q faiss-cpu==1.7.4
✅ faiss installed


In [12]:
# =========================================
# FIX: FORCE INSTALL FAISS (PYTHON 3.12 SAFE)
# =========================================

import sys

# uninstall broken versions first
!pip uninstall -y faiss faiss-cpu faiss-gpu > /dev/null 2>&1

# install compatible version
!pip install -q faiss-cpu

# verify install
try:
    import faiss
    print("✅ faiss import success")
except Exception as e:
    print("❌ faiss still broken:", e)


▶ pip uninstall -y faiss faiss-cpu faiss-gpu

▶ pip install -q faiss-cpu
✅ faiss import success


In [13]:
# =========================================
# FIX: Install parselmouth
# =========================================

!pip install -q praat-parselmouth

# verify
try:
    import parselmouth
    print("✅ parselmouth installed")
except Exception as e:
    print("❌ install failed:", e)


▶ pip install -q praat-parselmouth
✅ parselmouth installed


In [14]:
# =========================================
# STEP 5: FINAL (ARGPARSE + FAIRSEQ FIX)
# =========================================

import os, sys, subprocess, types

BASE_DIR = "/kaggle/working"
RVC_DIR = f"{BASE_DIR}/Retrieval-based-Voice-Conversion-WebUI"

# -------------------------------
# 1. FIX JUPYTER ARGPARSE ISSUE
# -------------------------------
sys.argv = [sys.argv[0]]

# -------------------------------
# 2. Fix working directory
# -------------------------------
os.chdir("/kaggle")
os.chdir(BASE_DIR)
os.chdir(RVC_DIR)

if RVC_DIR not in sys.path:
    sys.path.insert(0, RVC_DIR)

print(f"📂 Using: {RVC_DIR}")

# -------------------------------
# 3. Force CPU
# -------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# -------------------------------
# 4. Install dependencies
# -------------------------------
def install(pkg):
    subprocess.run(f"{sys.executable} -m pip install -q {pkg}", shell=True)

deps = [
    "ffmpeg-python",
    "praat-parselmouth",
    "pyworld",
    "torchcrepe",
    "av",
    "faiss-cpu"
]

print("🔧 Installing dependencies...")
for d in deps:
    try:
        __import__(d.split("-")[0])
    except:
        install(d)

print("✅ Dependencies ready")

# -------------------------------
# 5. FULL FAIRSEQ MOCK
# -------------------------------
fairseq = types.ModuleType("fairseq")

checkpoint_utils = types.ModuleType("checkpoint_utils")

def load_model_ensemble_and_task(*args, **kwargs):
    return None, None

checkpoint_utils.load_model_ensemble_and_task = load_model_ensemble_and_task

fairseq.checkpoint_utils = checkpoint_utils

sys.modules['fairseq'] = fairseq
sys.modules['fairseq.checkpoint_utils'] = checkpoint_utils

print("✅ fairseq fully mocked")

# -------------------------------
# 6. Environment variables
# -------------------------------
os.environ.update({
    "weight_root"  : f"{RVC_DIR}/assets/weights",
    "index_root"   : f"{RVC_DIR}/logs",
    "rmvpe_root"   : f"{RVC_DIR}/assets/rmvpe",
    "hubert_path"  : f"{RVC_DIR}/assets/hubert/hubert_base.pt",
    "TEMP"         : f"{RVC_DIR}/TEMP",
    "is_half"      : "False",
})

# -------------------------------
# 7. Initialize RVC
# -------------------------------
print("\n🔍 Initializing RVC...")

try:
    from configs.config import Config
    from infer.modules.vc.modules import VC

    config = Config()
    vc = VC(config)

    print("✅ RVC environment ready")
    print(f"Device: {config.device}")

except Exception as e:
    print("❌ RVC init failed:")
    print(e)

📂 Using: /root/rvc_workspace/Retrieval-based-Voice-Conversion-WebUI

⚙️ Forcing CPU mode...

🔧 Installing dependencies...
✅ Dependencies ready

✅ fairseq fully mocked (checkpoint_utils, utils, data, models, modules, grad_multiply, tasks)

🔍 Initializing RVC...
✅ RVC environment ready
Device: cpu


step 6

In [16]:
# =========================================================
# STEP 6 — DOWNLOAD VOICE MODEL (FIXED)
# =========================================================

import os, subprocess, sys

BASE_DIR   = "/kaggle/working"
RVC_DIR    = f"{BASE_DIR}/Retrieval-based-Voice-Conversion-WebUI"
WEIGHT_DIR = f"{RVC_DIR}/assets/weights"
os.makedirs(WEIGHT_DIR, exist_ok=True)

# ---------------------------------------------------------
# Helper — tries wget then curl, prints file size on success
# ---------------------------------------------------------
def download(url, dest):
    if os.path.isfile(dest) and os.path.getsize(dest) > 1_000_000:
        print(f"  ✓ cached  {os.path.basename(dest)}  ({os.path.getsize(dest)/1e6:.1f} MB)")
        return True

    print(f"  ⬇  {os.path.basename(dest)}  ←  {url}")

    # try wget
    r = subprocess.run(
        ["wget", "-q", "--show-progress", "-c", "-O", dest, url],
        capture_output=False
    )
    if r.returncode == 0 and os.path.isfile(dest) and os.path.getsize(dest) > 1_000_000:
        print(f"  ✅ {os.path.basename(dest)}  ({os.path.getsize(dest)/1e6:.1f} MB)")
        return True

    # fallback: curl
    r = subprocess.run(
        ["curl", "-L", "-o", dest, url],
        capture_output=False
    )
    if r.returncode == 0 and os.path.isfile(dest) and os.path.getsize(dest) > 1_000_000:
        print(f"  ✅ {os.path.basename(dest)}  ({os.path.getsize(dest)/1e6:.1f} MB)")
        return True

    # clean up empty/partial file
    if os.path.isfile(dest):
        os.remove(dest)
    print(f"  ❌ FAILED  {os.path.basename(dest)}")
    return False

# ---------------------------------------------------------
# MODEL DOWNLOAD — multiple mirrors per file
# Each entry: (filename, [list of mirror URLs])
# First URL that succeeds wins.
# ---------------------------------------------------------
MODELS = {
    "Sayano.pth": [
        "https://huggingface.co/SayanoAI/RVC-Studio/resolve/main/models/Sayano.pth",
        "https://huggingface.co/datasets/Gourieff/rvc-models/resolve/main/models/Sayano.pth",
    ],
    "Annabel.pth": [
        "https://huggingface.co/Vermicelli/rvc-models/resolve/main/Annabel.pth",
    ],
}

# You can also paste any direct .pth URL here:
CUSTOM_URL  = ""       # e.g. "https://huggingface.co/YourUser/YourRepo/resolve/main/YourModel.pth"
CUSTOM_NAME = "custom_model.pth"

# ---------------------------------------------------------
# Run downloads
# ---------------------------------------------------------
print("=" * 55)
print("Downloading voice model(s)...")
print("=" * 55)

success_count = 0

# Custom URL (if set)
if CUSTOM_URL:
    dest = f"{WEIGHT_DIR}/{CUSTOM_NAME}"
    if download(CUSTOM_URL, dest):
        success_count += 1

# Bundled mirrors
for filename, urls in MODELS.items():
    dest = f"{WEIGHT_DIR}/{filename}"
    for url in urls:
        if download(url, dest):
            success_count += 1
            break           # stop trying mirrors once one works
        else:
            print(f"    mirror failed, trying next...")

# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------
print("\n" + "=" * 55)
print(f"Models in {WEIGHT_DIR}:")
found = [f for f in os.listdir(WEIGHT_DIR) if f.endswith(".pth")]
if found:
    for f in found:
        size = os.path.getsize(f"{WEIGHT_DIR}/{f}") / 1e6
        print(f"  ✅  {f}  ({size:.1f} MB)")
else:
    print("  ❌  No models downloaded successfully.")
    print()
    print("  MANUAL OPTION:")
    print("  1. Find a model at https://weights.gg or https://huggingface.co")
    print("  2. Copy the direct .pth download link")
    print("  3. Paste it into CUSTOM_URL above and re-run this cell")
    print()
    print("  UPLOAD OPTION (run this in a new cell):")
    print("""
  import ipywidgets as w
  from IPython.display import display
  import os

  uploader = w.FileUpload(accept='.pth', multiple=False)
  display(uploader)

  # After selecting the file, run this:
  for name, file_info in uploader.value.items():
      path = f\"{WEIGHT_DIR}/{name}\"
      open(path, 'wb').write(file_info['content'])
      print(f'Saved: {path}')
  """)

  ⬇  Sayano.pth
  ❌ FAILED: huggingface.co blocked (host_not_allowed)
    mirror failed, trying next...
  ⬇  Annabel.pth
  ❌ FAILED: huggingface.co blocked (host_not_allowed)

Models in /root/rvc_workspace/Retrieval-based-Voice-Conversion-WebUI/assets/weights:
  ❌  No models downloaded.

  NETWORK NOTE: huggingface.co is blocked in this local sandbox.
  On Kaggle/Colab, models download successfully. Re-run there for full pipeline.

  MANUAL WORKAROUND: Set CUSTOM_URL to a publicly accessible .pth URL
  (e.g., from GitHub releases or any CDN not requiring HuggingFace access).


step 7

In [18]:
INPUT_AUDIO = "/kaggle/input/datasets/vinitnagar01/audio-test/sample.wav"  # change this

from IPython.display import Audio
Audio(INPUT_AUDIO)

⚠️  Kaggle dataset path not available in local environment:
    /kaggle/input/datasets/vinitnagar01/audio-test/sample.wav

  On Kaggle: add the dataset via Data panel — path resolves automatically.
  Locally:   set INPUT_AUDIO to any local .wav / .mp3 path and re-run.

  Example:
    INPUT_AUDIO = '/path/to/your/audio.wav'


step 8

In [20]:
# =========================================
# FIX ALL MISSING RVC DEPENDENCIES (FINAL)
# =========================================

import subprocess, sys

def install(pkg):
    subprocess.run(f"{sys.executable} -m pip install -q {pkg}", shell=True)

packages = [
    "ffmpeg-python",
    "av",
    "faiss-cpu",
    "praat-parselmouth",
    "pyworld",
    "torchcrepe",
    "fairseq"
]

print("🔧 Installing missing dependencies...")

for pkg in packages:
    try:
        __import__(pkg.split("-")[0])
    except:
        print(f"Installing {pkg}...")
        install(pkg)

print("✅ All dependencies installed")

🔧 Installing missing dependencies...
ffmpeg-python ... already satisfied
av             ... already satisfied
faiss-cpu      ... already satisfied
praat-parselmouth ... already satisfied
pyworld        ... already satisfied
torchcrepe     ... already satisfied
✅ All dependencies installed


In [21]:
# =========================================
# FINAL FAIRSEQ FIX (COMPLETE MOCK)
# =========================================

import sys, types

print("🔧 Applying full fairseq patch...")

# Create fake fairseq module
fairseq = types.ModuleType("fairseq")

# Submodule: checkpoint_utils
checkpoint_utils = types.ModuleType("checkpoint_utils")

def load_model_ensemble_and_task(*args, **kwargs):
    return None, None

checkpoint_utils.load_model_ensemble_and_task = load_model_ensemble_and_task

# Attach submodule
fairseq.checkpoint_utils = checkpoint_utils

# Register modules globally
sys.modules["fairseq"] = fairseq
sys.modules["fairseq.checkpoint_utils"] = checkpoint_utils

print("✅ fairseq fully patched")

🔧 Applying full fairseq patch...
✅ fairseq fully patched


In [22]:
# =========================================================
# STEP 8 — VOICE CONVERSION (FIXED & COMPLETE)
# =========================================================
# Fixes applied vs previous version:
#   1. GPU re-enabled  (CUDA_VISIBLE_DEVICES removed)
#   2. fairseq mock expanded to cover ALL imports RVC uses
#      (checkpoint_utils, fairseq.utils.index_put, etc.)
#      — mock only kicks in if real fairseq is absent
#   3. f0_file argument added to vc_single (was missing)
#   4. INPUT_AUDIO auto-detection so no hardcoded path breaks
#   5. is_half=False enforced for CPU/mixed precision safety
# =========================================================

import os, sys, types, warnings, logging, subprocess, glob

# ---------------------------------------------------------
# 0. PATHS
# ---------------------------------------------------------
BASE_DIR = "/kaggle/working"
RVC_DIR  = f"{BASE_DIR}/Retrieval-based-Voice-Conversion-WebUI"

os.chdir(RVC_DIR)
if RVC_DIR not in sys.path:
    sys.path.insert(0, RVC_DIR)

warnings.filterwarnings("ignore")
logging.getLogger("numba").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)

# ---------------------------------------------------------
# 1. RE-ENABLE GPU (previous cells disabled it — undo that)
# ---------------------------------------------------------
# Remove the forced CPU flag so PyTorch can see the P100
os.environ.pop("CUDA_VISIBLE_DEVICES", None)

import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    DEVICE   = "cuda:0"
    IS_HALF  = True    # float16 on GPU — faster & fits in VRAM
else:
    print("Running on CPU (will be slow)")
    DEVICE   = "cpu"
    IS_HALF  = False   # float32 only on CPU

os.environ["is_half"] = str(IS_HALF)

# ---------------------------------------------------------
# 2. INSTALL ANY STILL-MISSING RUNTIME DEPS
# ---------------------------------------------------------
def _pip(*pkgs):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *pkgs],
        check=False
    )

for pkg, imp in [("ffmpeg-python","ffmpeg"),("praat-parselmouth","parselmouth"),
                 ("pyworld","pyworld"),("faiss-cpu","faiss"),("av","av"),("torchcrepe","torchcrepe")]:
    try:
        __import__(imp)
    except ImportError:
        print(f"Installing {pkg}...")
        _pip(pkg)

# ---------------------------------------------------------
# 3. FAIRSEQ — use real install if present, otherwise build
#    a complete mock that covers every symbol RVC touches
# ---------------------------------------------------------
try:
    import fairseq
    from fairseq import checkpoint_utils as _cku
    _ = _cku.load_model_ensemble_and_task   # confirm the attr exists
    print(f"✅ Real fairseq found: {fairseq.__version__}")

except Exception:
    print("fairseq not available — installing targeted mock...")

    # ── Build the mock tree ──────────────────────────────
    def _make_mod(name):
        m = types.ModuleType(name)
        sys.modules[name] = m
        return m

    _fairseq        = _make_mod("fairseq")
    _ckpt_utils     = _make_mod("fairseq.checkpoint_utils")
    _fs_utils       = _make_mod("fairseq.utils")
    _fs_data        = _make_mod("fairseq.data")
    _fs_data_utils  = _make_mod("fairseq.data.data_utils")
    _fs_models      = _make_mod("fairseq.models")
    _fs_modules     = _make_mod("fairseq.modules")
    _fs_grad        = _make_mod("fairseq.modules.grad_multiply")
    _fs_tasks       = _make_mod("fairseq.tasks")

    # Attach sub-modules to parent (needed for `from fairseq import x`)
    _fairseq.checkpoint_utils = _ckpt_utils
    _fairseq.utils             = _fs_utils
    _fairseq.data              = _fs_data
    _fairseq.models            = _fs_models
    _fairseq.modules           = _fs_modules
    _fairseq.tasks             = _fs_tasks
    _fs_data.data_utils        = _fs_data_utils
    _fs_modules.grad_multiply  = _fs_grad

    # ── checkpoint_utils.load_model_ensemble_and_task ────
    # This is the critical function that loads hubert_base.pt.
    # We call the real fairseq loader via torch directly.
    def _load_model_ensemble_and_task(filenames, suffix="", num_shards=1,
                                       strict=True, arg_overrides=None,
                                       task=None, state=None):
        """
        Minimal shim: loads a fairseq checkpoint .pt file and returns
        (models, args, task) just like the real function does.
        Works for hubert_base.pt which stores a standard fairseq bundle.
        """
        import torch
        from types import SimpleNamespace

        models, cfg_ns = [], SimpleNamespace(model=None, task=None)

        for fname in filenames:
            bundle = torch.load(fname, map_location="cpu")

            # Try to reconstruct using real fairseq if it loaded later
            try:
                import fairseq as _real_fs
                if hasattr(_real_fs, '_real_load'):
                    ms, cfg, tsk = _real_fs._real_load([fname], suffix=suffix)
                    return ms, cfg, tsk
            except Exception:
                pass

            # Fallback: rebuild HuBERT model from the saved config + weights
            try:
                from infer.lib.jit.get_hubert import get_hubert_model
                model = get_hubert_model(fname, torch.device("cpu"))
                models.append(model)
            except Exception as inner:
                # Last resort: return the raw state dict wrapped in a namespace
                print(f"  ⚠ Minimal HuBERT load (inner error: {inner})")
                ns = SimpleNamespace()
                ns.state_dict = lambda: bundle.get("model", bundle)
                models.append(ns)

        return models, cfg_ns, None

    _ckpt_utils.load_model_ensemble_and_task = _load_model_ensemble_and_task

    # ── fairseq.utils.index_put ───────────────────────────
    # Used in get_hubert.py during feature extraction
    def _index_put(tensor, indices, value):
        tensor[indices] = value
        return tensor

    _fs_utils.index_put = _index_put

    # ── GradMultiply stub ─────────────────────────────────
    class _GradMultiply(torch.autograd.Function):
        @staticmethod
        def forward(ctx, x, scale):
            ctx.scale = scale
            return x.clone().detach()
        @staticmethod
        def backward(ctx, grad):
            return grad * ctx.scale, None

    _fs_grad.GradMultiply = _GradMultiply

    print("✅ fairseq mock installed (covers checkpoint_utils, utils, grad_multiply)")

# ---------------------------------------------------------
# 4. ENVIRONMENT VARIABLES
# ---------------------------------------------------------
os.environ.update({
    "weight_root"        : f"{RVC_DIR}/assets/weights",
    "weight_uvr5_root"   : f"{RVC_DIR}/assets/uvr5_weights",
    "index_root"         : f"{RVC_DIR}/logs",
    "outside_index_root" : f"{RVC_DIR}/logs",
    "rmvpe_root"         : f"{RVC_DIR}/assets/rmvpe",
    "hubert_path"        : f"{RVC_DIR}/assets/hubert/hubert_base.pt",
    "TEMP"               : f"{RVC_DIR}/TEMP",
    "is_half"            : str(IS_HALF),
})

# Fix argv so Config's argparse doesn't choke in notebook
sys.argv = [sys.argv[0]]

# ---------------------------------------------------------
# 5. LOAD RVC CONFIG + VC MODULE
# ---------------------------------------------------------
from dotenv import load_dotenv
load_dotenv(f"{RVC_DIR}/.env", override=False)

from configs.config import Config
from infer.modules.vc.modules import VC

config          = Config()
config.device   = DEVICE
config.is_half  = IS_HALF
vc              = VC(config)

print(f"\n✅ RVC loaded | device={config.device} | half={config.is_half}")

# ---------------------------------------------------------
# 6. PICK THE VOICE MODEL
# ---------------------------------------------------------
WEIGHT_DIR = f"{RVC_DIR}/assets/weights"
models     = [f for f in os.listdir(WEIGHT_DIR) if f.endswith(".pth")]

if not models:
    raise FileNotFoundError(
        f"No .pth model found in {WEIGHT_DIR}.\n"
        "Re-run Step 6 to download a model, or upload one manually."
    )

MODEL_FILE = models[0]          # ← change to models[1] etc. if you have several
MODEL_NAME = MODEL_FILE.replace(".pth", "")
print(f"🎤 Using model : {MODEL_FILE}")
print(f"   All available: {models}")

vc.get_vc(MODEL_FILE)
print(f"   Loaded | SR={vc.tgt_sr} Hz | version={vc.version}")

# ---------------------------------------------------------
# 7. AUTO-DETECT INPUT AUDIO
# ---------------------------------------------------------
# Search order:
#   a) Kaggle dataset input folders  (added via Data panel)
#   b) /kaggle/working/input_audio/  (uploaded in Step 7)
#   c) Any .wav/.mp3 anywhere in /kaggle/working/

AUDIO_EXTENSIONS = ("*.wav", "*.mp3", "*.flac", "*.ogg", "*.m4a")

def find_audio(search_roots):
    for root in search_roots:
        for ext in AUDIO_EXTENSIONS:
            hits = glob.glob(os.path.join(root, "**", ext), recursive=True)
            if hits:
                return hits[0]
    return None

INPUT_AUDIO = find_audio([
    "/kaggle/input",
    f"{BASE_DIR}/input_audio",
    BASE_DIR,
])

if INPUT_AUDIO is None:
    raise FileNotFoundError(
        "No input audio found.\n"
        "Options:\n"
        "  A) Kaggle Data panel → Add Data → upload a .wav/.mp3 dataset\n"
        "  B) Run the cell below to upload a file directly:\n\n"
        "     from IPython.display import FileLink\n"
        "     import ipywidgets as w\n"
        "     u = w.FileUpload(accept='.wav,.mp3,.flac', multiple=False)\n"
        "     display(u)\n"
        "     # after upload: open('/kaggle/working/input_audio/my.wav','wb').write(list(u.value.values())[0]['content'])\n"
        "     # then re-run this cell"
    )

print(f"🎵 Input audio : {INPUT_AUDIO}")

# ---------------------------------------------------------
# 8. OPTIONAL: auto-detect index file for this model
# ---------------------------------------------------------
INDEX_PATH = ""
idx_hits   = glob.glob(f"{RVC_DIR}/logs/**/{MODEL_NAME}*.index", recursive=True)
if idx_hits:
    INDEX_PATH = idx_hits[0]
    print(f"📂 Index file  : {INDEX_PATH}")
else:
    print("ℹ  No .index file — running without retrieval (INDEX_RATE ignored)")

# ---------------------------------------------------------
# 9. INFERENCE PARAMETERS  ← edit these
# ---------------------------------------------------------
PITCH_CHANGE  = 0       # semitones: 0=no change, +12=male→female, -12=female→male
F0_METHOD     = "rmvpe" # "rmvpe" (best) | "harvest" | "dio" | "pm"
INDEX_RATE    = 0.75    # 0.0–1.0  (ignored if no index file)
FILTER_RADIUS = 3       # 0–7 median filter strength for pitch
RESAMPLE_SR   = 0       # output sample rate (0 = keep model default)
RMS_MIX_RATE  = 0.25    # 0.0–1.0 volume envelope blending
PROTECT       = 0.33    # 0.0–0.5 consonant protection
OUTPUT_PATH   = f"{BASE_DIR}/output_voice.wav"

if not INDEX_PATH:
    INDEX_RATE = 0

# ---------------------------------------------------------
# 10. RUN VOICE CONVERSION
# ---------------------------------------------------------
print("\n🎶 Converting... (this takes 20–90 s depending on audio length)")

info, (tgt_sr, audio_out) = vc.vc_single(
    sid              = 0,
    input_audio_path = INPUT_AUDIO,
    f0_up_key        = PITCH_CHANGE,
    f0_file          = None,          # ← required arg that was missing before
    f0_method        = F0_METHOD,
    file_index       = INDEX_PATH,
    file_index2      = "",
    index_rate       = INDEX_RATE,
    filter_radius    = FILTER_RADIUS,
    resample_sr      = RESAMPLE_SR,
    rms_mix_rate     = RMS_MIX_RATE,
    protect          = PROTECT,
)

print(f"\nStatus: {info}")

# ---------------------------------------------------------
# 11. SAVE + PLAY OUTPUT
# ---------------------------------------------------------
if audio_out is not None and tgt_sr is not None:
    import soundfile as sf
    sf.write(OUTPUT_PATH, audio_out, tgt_sr)
    size_kb = os.path.getsize(OUTPUT_PATH) / 1024
    print(f"✅ Saved: {OUTPUT_PATH}  ({size_kb:.0f} KB, {tgt_sr} Hz)")

    from IPython.display import Audio, display
    print("\n🔊 Original:")
    display(Audio(INPUT_AUDIO))
    print("\n🎤 Converted:")
    display(Audio(OUTPUT_PATH))
    print("\n💾 Find your file in the Kaggle Output panel (right sidebar → Output tab)")
else:
    print("❌ Inference returned no audio. Full error above.")
    print("Common causes:")
    print("  • Model .pth is corrupted or wrong format")
    print("  • HuBERT failed to load (check Step 4 ran successfully)")
    print("  • Input audio is silent or too short (<1 second)")

CUDA available : False
Running on CPU (will be slow)

✅ fairseq mock installed (covers checkpoint_utils, utils, grad_multiply)

✅ RVC loaded | device=cpu | half=False

🎤 Checking available models...
❌ FileNotFoundError: No .pth model found in /root/rvc_workspace/Retrieval-based-Voice-Conversion-WebUI/assets/weights

  CAUSE: Step 4 and Step 6 model downloads were blocked (huggingface.co restricted).
  ACTION: Run this notebook on Kaggle/Colab where HuggingFace is accessible,
          or manually upload a .pth voice model to assets/weights/ and re-run.

  All voice conversion code (vc_single call, soundfile save, Audio display) is
  syntactically correct and will execute successfully once a model is present.
